# Competition Analysis — AI Actors Network

Analyse concurrentielle à partir des données de la base SQLite :
1. Export des couples entreprise / concurrent
2. Format long + agrégation
3. Matrice dirigée entreprise × compétiteurs
4. Projection 2D (t-SNE)

Population analysée : sélection par capitalisation, et si absente par fonds levés.
Score de ranking : capitalisation, sinon fonds levés.

## 1. Configuration et imports

In [12]:
import sqlite3
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE
from sklearn.preprocessing import normalize

ROOT        = Path(__file__).parent.parent if "__file__" in dir() else Path().resolve().parent
DB_PATH     = ROOT / "database.db"
EXPORTS_DIR = ROOT / "analyses" / "exports"
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

COL_NAME        = "name"
COL_SECTOR      = "sector"
COL_COMPETITORS = "main_competitors"
RANDOM_SEED     = 42

print(f"Base : {DB_PATH}")
print(f"Exports : {EXPORTS_DIR}")

Base : C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\database.db
Exports : C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports


## 2. Extraction SQL — entreprise / secteurs / concurrents

In [75]:
SQL = f"""
WITH base AS (
    SELECT
        name             AS {COL_NAME},
        sector           AS {COL_SECTOR},
        main_competitors AS {COL_COMPETITORS},
        CASE
            WHEN NULLIF(TRIM(capitalization), '') IS NULL THEN NULL
            ELSE CAST(REPLACE(capitalization, ',', '.') AS REAL)
        END AS cap_musd,
        CASE
            WHEN NULLIF(TRIM(funds_raised), '') IS NULL THEN NULL
            ELSE CAST(REPLACE(funds_raised, ',', '.') AS REAL)
        END AS funds_musd
    FROM enterprises
    WHERE main_competitors IS NOT NULL
      AND main_competitors != ''
)
SELECT
    {COL_NAME},
    {COL_SECTOR},
    {COL_COMPETITORS},
    COALESCE(cap_musd, funds_musd) AS ranking_score
FROM base
WHERE COALESCE(cap_musd, funds_musd) IS NOT NULL
  AND COALESCE(cap_musd, funds_musd) > 0
ORDER BY ranking_score DESC
"""

with sqlite3.connect(DB_PATH) as con:
    df_raw = pd.read_sql_query(SQL, con)

print(
    f"{len(df_raw)} entreprises retenues "
    f"(sélection: capitalisation, sinon fonds levés)"
)
df_raw[["name", "ranking_score"]].head(10)

83 entreprises retenues (sélection: capitalisation, sinon fonds levés)


,name,ranking_score
0,AMD,794.0
1,PolyAI,750.0
2,Y combinator,600.0
3,ByteDance,550.0
4,Intel,508.0
5,Cisco,465.0
6,Oracle,408.0
7,YouTube,400.0
8,Palantir,318.0
9,Alibaba,282.0


In [76]:
# Export brut
raw_path = EXPORTS_DIR / "competitors_raw.csv"
df_raw.to_csv(raw_path, index=False)
print(f"Export brut → {raw_path}")

Export brut → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_raw.csv


## 3. Nettoyage et standardisation des noms de concurrents

In [77]:
_SEP = re.compile(r"[,;/\n]+")
_INVALID = re.compile(r"^(na|n/a|none|unknown|tbd|-)$", re.IGNORECASE)

def clean_name(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\(.*?\)", "", s).strip()
    s = re.sub(r"\s{2,}", " ", s)
    return s.title()

def split_competitors(raw: str) -> list[str]:
    parts = _SEP.split(str(raw))
    return [clean_name(p) for p in parts
            if clean_name(p) and not _INVALID.match(p.strip())]

df_clean = df_raw.copy()
df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(split_competitors)

df_clean[COL_COMPETITORS] = df_clean.apply(
    lambda r: [c for c in r[COL_COMPETITORS] if c.lower() != r[COL_NAME].lower()],
    axis=1,
)

df_clean = df_clean[df_clean[COL_COMPETITORS].map(len) > 0].reset_index(drop=True)

print(f"{len(df_clean)} entreprises après nettoyage")
df_clean[[COL_NAME, COL_COMPETITORS]].head(8)

83 entreprises après nettoyage


,name,main_competitors
0,AMD,"[Nvidia, Broadcom, Marvell, Intel, Qualcomm, A..."
1,PolyAI,"[Bland Ai, Vapi, Retell Ai, Parloa, Ibm, Googl..."
2,Y combinator,"[Techstars, 500 Global, Sequoia, Horowitz, Ant..."
3,ByteDance,"[Meta, Youtube, Tencent, Openai, Google Deepmind]"
4,Intel,"[Ams, Qualcomm, Nvidia, Tsmc, Samsung Electron..."
5,Cisco,"[Arista Networks, Juniper Networks, Huawei, No..."
6,Oracle,"[Amazon, Google, Microsoft, Sap, Saleforce]"
7,YouTube,"[Tiktok, Instagram, Twitch, Vimeo, Deezer]"


## 3b. Normalisation sémantique — filiales → groupe parent

In [78]:
SEMANTIC_ALIASES: dict[str, str] = {
    # ── Google (nom canonique dans la base) ───────────────────────────────────
    "Alphabet":             "Google",
    "Alphabet Inc.":        "Google",
    "Youtube":              "Google",
    "YouTube":              "Google",
    "Deepmind":             "Google",
    "DeepMind":             "Google",
    "Google Deepmind":      "Google",
    "Google DeepMind":      "Google",
    "Google Brain":         "Google",
    "Google Cloud":         "Google",
    "Google Cloud Platform":"Google",
    "GCP":                  "Google",
    "Waymo":                "Google",
    "Verily":               "Google",
    "Calico":               "Google",
    "Waze":                 "Google",
    "Google Translate":     "Google",
    "Gmail":                "Google",
    "Android":              "Google",
    # ── Meta (nom canonique dans la base) ─────────────────────────────────────
    "Meta Platforms":       "Meta",
    "Facebook":             "Meta",
    "Instagram":            "Meta",
    "Whatsapp":             "Meta",
    "WhatsApp":             "Meta",
    "Threads":              "Meta",
    "Oculus":               "Meta",
    "Meta Quest":           "Meta",
    "LLaMA":                "Meta",
    "Llama":                "Meta",
    # ── Microsoft ─────────────────────────────────────────────────────────────
    "Azure":                "Microsoft",
    "Microsoft Azure":      "Microsoft",
    "Linkedin":             "Microsoft",
    "LinkedIn":             "Microsoft",
    "Github":               "Microsoft",
    "GitHub":               "Microsoft",
    "Skype":                "Microsoft",
    "Bing":                 "Microsoft",
    "Nuance":               "Microsoft",
    "Nuance Communications":"Microsoft",
    "Activision Blizzard":  "Microsoft",
    "Activision":           "Microsoft",
    "Xbox":                 "Microsoft",
    "Microsoft Translator": "Microsoft",
    "Office 365":           "Microsoft",
    # ── Amazon ────────────────────────────────────────────────────────────────
    "Aws":                  "Amazon",
    "AWS":                  "Amazon",
    "Alexa":                "Amazon",
    "Twitch":               "Amazon",
    "Amazon.Com":           "Amazon",
    "Amazon Prime":         "Amazon",
    "Amazon Prime Video":   "Amazon",
    "Kindle":               "Amazon",
    # ── Amazon Web Services (entité séparée dans la base) ─────────────────────
    "Amazon Web Services (AWS)": "Amazon Web Services",
    # ── Apple ─────────────────────────────────────────────────────────────────
    "Siri":                 "Apple",
    "Apple Inc.":           "Apple",
    "Apple Inc":            "Apple",
    "Iphone":               "Apple",
    "iPhone":               "Apple",
    "Ipad":                 "Apple",
    "iPad":                 "Apple",
    "Apple Silicon":        "Apple",
    # ── Salesforce ────────────────────────────────────────────────────────────
    "Slack":                "Salesforce",
    "Tableau":              "Salesforce",
    "Mulesoft":             "Salesforce",
    "MuleSoft":             "Salesforce",
    "Salesforce - Einstein":"Salesforce",
    "Einstein":             "Salesforce",
    # ── IBM ───────────────────────────────────────────────────────────────────
    "Red Hat":              "IBM",
    "RedHat":               "IBM",
    "Watsonx":              "IBM",
    "Watson":               "IBM",
    "IBM Watson":           "IBM",
    # ── Oracle ────────────────────────────────────────────────────────────────
    "Netsuite":             "Oracle",
    "NetSuite":             "Oracle",
    "Java":                 "Oracle",
    # ── Nvidia (nom canonique dans la base) ───────────────────────────────────
    "NVIDIA":               "Nvidia",
    "Cuda":                 "Nvidia",
    "CUDA":                 "Nvidia",
    "Nvidia Corporation":   "Nvidia",
    # ── INtel (nom avec cette casse dans la base) ─────────────────────────────
    "Intel":                "INtel",
    "Intel Corporation":    "INtel",
    # ── AMD ───────────────────────────────────────────────────────────────────
    "AMD Inc.":             "AMD",
    "Advanced Micro Devices": "AMD",
    # ── ByteDance ─────────────────────────────────────────────────────────────
    "Tiktok":               "ByteDance",
    "TikTok":               "ByteDance",
    "Douyin":               "ByteDance",
    "Bytedance":            "ByteDance",
    # ── X (nom canonique dans la base, anciennement Twitter) ──────────────────
    "Twitter":              "X",
    "X.Com":                "X",
    "X Corp":               "X",
    # ── Tesla ─────────────────────────────────────────────────────────────────
    "Tesla Inc.":           "Tesla",
    "Tesla Motors":         "Tesla",
    # ── SpaceX ────────────────────────────────────────────────────────────────
    "Space Exploration Technologies": "SpaceX",
    "Starlink":             "SpaceX",
    # ── Baidu ─────────────────────────────────────────────────────────────────
    "Ernie":                "Baidu",
    "Ernie Bot":            "Baidu",
    "ERNIE":                "Baidu",
    # ── Tencent ───────────────────────────────────────────────────────────────
    "Wechat":               "Tencent",
    "WeChat":               "Tencent",
    "Qq":                   "Tencent",
    "QQ":                   "Tencent",
    # ── OpenAI ────────────────────────────────────────────────────────────────
    "Chatgpt":              "OpenAI",
    "ChatGPT":              "OpenAI",
    "Gpt-4":                "OpenAI",
    "GPT-4":                "OpenAI",
    "Gpt4":                 "OpenAI",
    "GPT4":                 "OpenAI",
    "Openai":               "OpenAI",
    "DALL-E":               "OpenAI",
    "Dall-E":               "OpenAI",
    "Sora":                 "OpenAI",
    # ── Anthropic ─────────────────────────────────────────────────────────────
    "Claude":               "Anthropic",
    "Claude AI":            "Anthropic",
    # ── Samsung ───────────────────────────────────────────────────────────────
    "Samsung Electronics":  "Samsung",
    "Samsung System LSI":   "Samsung",
    # ── Adobe ─────────────────────────────────────────────────────────────────
    "Adobe Firefly":        "Adobe",
    "Photoshop":            "Adobe",
    # ── Qualcomm ──────────────────────────────────────────────────────────────
    "Qualcomm Inc.":        "Qualcomm",
    # ── SAP ───────────────────────────────────────────────────────────────────
    "SAP SE":               "SAP",
    # ── Huawei ────────────────────────────────────────────────────────────────
    "Huawei Technologies":  "Huawei",
    # ── Alibaba ───────────────────────────────────────────────────────────────
    "Alibaba Group":        "Alibaba",
    "AliCloud":             "Alibaba",
    "Alipay":               "Alibaba",
    # ── HP ────────────────────────────────────────────────────────────────────
    "Hewlett-Packard":      "HP",
    "Hewlett Packard":      "HP",
    # ── Netflix ───────────────────────────────────────────────────────────────
    "Netflix Inc.":         "Netflix",
    # ── Spotify ───────────────────────────────────────────────────────────────
    "Spotify AB":           "Spotify",
    # ── Sony ──────────────────────────────────────────────────────────────────
    "Sony Corporation":     "Sony",
    "PlayStation":          "Sony",
    # ── Dell ──────────────────────────────────────────────────────────────────
    "Dell Technologies":    "Dell",
    # ── Lenovo ────────────────────────────────────────────────────────────────
    "Lenovo Group":         "Lenovo",
    # ── MediaTek ──────────────────────────────────────────────────────────────
    "MediaTek Inc.":        "MediaTek",
    # ── Xiaomi ────────────────────────────────────────────────────────────────
    "Xiaomi Corporation":   "Xiaomi",
    # ── Oppo ──────────────────────────────────────────────────────────────────
    "OPPO":                 "Oppo",
    # ── Vivo ──────────────────────────────────────────────────────────────────
    "VIVO":                 "Vivo",
    # ── Disney ────────────────────────────────────────────────────────────────
    "Disney+":              "Disney",
    "Walt Disney":          "Disney",
    # ── Boeing ────────────────────────────────────────────────────────────────
    "Boeing Company":       "Boeing",
    # ── Autres acteurs spécifiques IA ─────────────────────────────────────────
    "Mistral":              "Mistral AI",
    "Stability AI":         "Stability",
    "Stability.ai":         "Stability",
    "Midjourney Inc.":      "Midjourney",
    "Runway ML":            "Runway",
}

def apply_semantic_aliases(names: list[str]) -> list[str]:
    return [SEMANTIC_ALIASES.get(n, n) for n in names]

df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(apply_semantic_aliases)
df_clean[COL_COMPETITORS] = df_clean[COL_COMPETITORS].apply(lambda lst: list(dict.fromkeys(lst)))
df_clean[COL_COMPETITORS] = df_clean.apply(
    lambda r: [c for c in r[COL_COMPETITORS]
               if c.lower() != r[COL_NAME].lower()
               and SEMANTIC_ALIASES.get(r[COL_NAME].title(), r[COL_NAME]).lower() != c.lower()],
    axis=1,
)
df_clean = df_clean[df_clean[COL_COMPETITORS].map(len) > 0].reset_index(drop=True)

# explode garde le nom COL_COMPETITORS, pas "competitor"
preview = df_clean[[COL_NAME, COL_COMPETITORS]].explode(COL_COMPETITORS)
print(f"{len(SEMANTIC_ALIASES)} aliases | {len(df_clean)} entreprises après normalisation sémantique")
print("\nTop concurrents après normalisation :")
print(preview[COL_COMPETITORS].value_counts().head(15).to_string())

143 aliases | 83 entreprises après normalisation sémantique

Top concurrents après normalisation :
main_competitors
Google       23
Microsoft    19
OpenAI       15
Meta         10
Amazon       10
Apple         9
Anthropic     9
Nvidia        7
ByteDance     7
INtel         6
Qualcomm      6
Tencent       5
Samsung       4
Amd           4
Oracle        4


## 4. Format long — couples entreprise→concurrent (asymétrique, sans filtre strict)

In [79]:
def _norm_key(value: str) -> str:
    return re.sub(r"\s+", " ", str(value).strip()).casefold()

# Référence de canonicalisation: tous les noms d'entreprises de la base.
with sqlite3.connect(DB_PATH) as con:
    df_all_names = pd.read_sql_query("SELECT name FROM enterprises WHERE name IS NOT NULL", con)

canonical_by_key = {}
for name in df_all_names["name"].tolist():
    clean = str(name).strip()
    if clean:
        key = _norm_key(clean)
        if key not in canonical_by_key:
            canonical_by_key[key] = clean

df_long = (
    df_clean
    .explode(COL_COMPETITORS)
    .rename(columns={COL_COMPETITORS: "competitor"})
    .reset_index(drop=True)
    [[COL_NAME, "competitor", COL_SECTOR, "ranking_score"]]
)

# Canonicalise les compétiteurs pour éviter les doublons de casse/écriture.
df_long["competitor"] = (
    df_long["competitor"]
    .map(lambda c: canonical_by_key.get(_norm_key(c), c))
)

# Dédoublonne au niveau entreprise→compétiteur (dataset dirigé).
df_long = (
    df_long.drop_duplicates(subset=[COL_NAME, "competitor"])
    .reset_index(drop=True)
)
df_long.insert(0, "pair_id", df_long.index)

long_path = EXPORTS_DIR / "competitors_long.csv"
df_long.to_csv(long_path, index=False)

print(f"{len(df_long)} couples entreprise→concurrent après normalisation et dédoublonnage")
print(f"Entreprises (lignes): {df_long[COL_NAME].nunique()} | Compétiteurs (colonnes potentielles): {df_long['competitor'].nunique()}")
print(f"Export → {long_path}")
df_long.head(10)

490 couples entreprise→concurrent après normalisation et dédoublonnage
Entreprises (lignes): 83 | Compétiteurs (colonnes potentielles): 320
Export → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_long.csv


,pair_id,name,competitor,sector,ranking_score
0,0,AMD,Nvidia,Hardware,794.0
1,1,AMD,Broadcom,Hardware,794.0
2,2,AMD,Marvell,Hardware,794.0
3,3,AMD,Intel,Hardware,794.0
4,4,AMD,Qualcomm,Hardware,794.0
5,5,AMD,Apple,Hardware,794.0
6,6,AMD,Google,Hardware,794.0
7,7,PolyAI,Bland Ai,"AI model, Sales & Marketing, Voice",750.0
8,8,PolyAI,Vapi,"AI model, Sales & Marketing, Voice",750.0
9,9,PolyAI,Retell Ai,"AI model, Sales & Marketing, Voice",750.0


## 5. Agrégation et comptage des relations concurrentielles

In [80]:
df_agg = (
    df_long
    .groupby([COL_NAME, "competitor"], sort=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 paires entreprise–concurrent :")
display(df_agg.head(20))

print("\nDistribution des fréquences :")
display(df_agg["count"].describe())

agg_path = EXPORTS_DIR / "competitors_aggregated.csv"
df_agg.to_csv(agg_path, index=False)
print(f"\nExport → {agg_path}")

Top 20 paires entreprise–concurrent :


,name,competitor,count
0,AMD,Nvidia,1
1,AMD,Broadcom,1
2,AMD,Marvell,1
3,AMD,Intel,1
4,AMD,Qualcomm,1
5,AMD,Apple,1
6,AMD,Google,1
7,PolyAI,Bland Ai,1
8,PolyAI,Vapi,1
9,PolyAI,Retell Ai,1



Distribution des fréquences :


count    490.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: count, dtype: float64


Export → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competitors_aggregated.csv


## 6. Matrice dirigée entreprise × compétiteurs

In [81]:
# Matrice dirigée: lignes = entreprises, colonnes = compétiteurs.
# On garde la logique asymétrique, sans M + M^T.
df_matrix = df_agg.pivot_table(
    index=COL_NAME,
    columns="competitor",
    values="count",
    fill_value=0,
)

enterprise_actors = sorted(df_raw[COL_NAME].dropna().astype(str).str.strip().unique())
df_matrix = df_matrix.reindex(index=enterprise_actors, fill_value=0)

print(f"Matrice dirigée: {df_matrix.shape[0]} entreprises × {df_matrix.shape[1]} compétiteurs")
print(f"Densité non-nulle : {(df_matrix.values > 0).mean():.1%}")

cooc_path = EXPORTS_DIR / "cooccurrence_matrix.csv"
df_matrix.to_csv(cooc_path)
print(f"Export matrice dirigée → {cooc_path}")

Matrice dirigée: 83 entreprises × 320 compétiteurs
Densité non-nulle : 1.8%
Export matrice dirigée → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\cooccurrence_matrix.csv


## 7. t-SNE sur la matrice entreprise × compétiteurs

Projection non linéaire des lignes (entreprises) dans l'espace des variables (compétiteurs).

In [82]:
N = df_matrix.shape[0]
X = normalize(df_matrix.values, norm="l2")

perplexity = 5
print(f"t-SNE 2D sur {N} entreprises (perplexity={perplexity})")
coords = TSNE(
    n_components=2,
    perplexity=perplexity,
    init="pca",
    learning_rate="auto",
    metric="cosine",
    random_state=RANDOM_SEED,
).fit_transform(X)
method = "t-SNE"

out_degree = df_matrix.sum(axis=1).to_dict()
ranking_score_map = df_raw.set_index(COL_NAME)["ranking_score"].to_dict()

enterprise_names = list(df_matrix.index)
df_coords = pd.DataFrame({"actor": enterprise_names, "x": coords[:, 0], "y": coords[:, 1]})
df_coords["score"] = df_coords["actor"].map(out_degree).fillna(0)
df_coords["log_score"] = np.log10(df_coords["score"] + 1)
df_coords["ranking_score"] = df_coords["actor"].map(ranking_score_map).fillna(0)

sector_map = (
    df_raw.set_index(COL_NAME)[COL_SECTOR]
    .dropna()
    .apply(lambda s: s.split(",")[0].strip())
    .to_dict()
)
df_coords["sector"] = df_coords["actor"].map(sector_map).fillna("Unknown")

coords_path = EXPORTS_DIR / "coords_2d.csv"
df_coords.to_csv(coords_path, index=False)
print(f"Coordonnées 2D ({method}) → {coords_path}")
df_coords.sort_values("ranking_score", ascending=False).head(10)

t-SNE 2D sur 83 entreprises (perplexity=5)
Coordonnées 2D (t-SNE) → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\coords_2d.csv


,actor,x,y,score,log_score,ranking_score,sector
4,AMD,6.985020,80.631462,7.0,0.903090,794.0,Hardware
59,PolyAI,6.650085,25.020899,6.0,0.845098,750.0,AI model
81,Y combinator,-178.388794,127.354012,6.0,0.845098,600.0,Unknown
24,ByteDance,-16.992659,-24.272873,4.0,0.698970,550.0,Media & Entertainment
44,Intel,12.881263,98.877983,5.0,0.778151,508.0,Hardware
27,Cisco,-0.792403,4.542050,10.0,1.041393,465.0,Unknown
53,Oracle,37.718838,-4.710128,5.0,0.778151,408.0,Cloud Provider
82,YouTube,32.488270,-40.977509,5.0,0.778151,400.0,Media & Entertainment
57,Palantir,48.221508,-5.089189,8.0,0.954243,318.0,Defence
9,Alibaba,41.101402,-42.342793,8.0,0.954243,282.0,Cloud Provider


In [38]:
def normalize_actor_key(value: str) -> str:
    return re.sub(r"\s+", " ", str(value).strip()).casefold()

if "country" in df_raw.columns:
    df_raw_country = df_raw[[COL_NAME, "country"]].copy()
else:
    with sqlite3.connect(DB_PATH) as con:
        df_raw_country = pd.read_sql_query(
            f"SELECT name AS {COL_NAME}, country FROM enterprises",
            con,
        )

enterprise_country = (
    df_raw_country
    .dropna(subset=[COL_NAME])
    .assign(
        actor=lambda d: d[COL_NAME].map(lambda v: SEMANTIC_ALIASES.get(str(v).strip(), str(v).strip())),
        actor_key=lambda d: d["actor"].map(normalize_actor_key),
    )
)
enterprise_country = enterprise_country[
    enterprise_country["country"].notna()
    & enterprise_country["country"].astype(str).str.strip().ne("")
].copy()
enterprise_country = enterprise_country.drop_duplicates(subset=["actor_key"], keep="first")

competitor_country = (
    df_long[[COL_NAME, "competitor"]]
    .rename(columns={COL_NAME: "source_enterprise", "competitor": "actor"})
    .copy()
)
competitor_country["actor"] = competitor_country["actor"].map(
    lambda v: SEMANTIC_ALIASES.get(str(v).strip(), str(v).strip())
)
competitor_country["actor_key"] = competitor_country["actor"].map(normalize_actor_key)
competitor_country = competitor_country.merge(
    enterprise_country[["actor_key", "actor", "country"]].rename(
        columns={"actor": "matched_enterprise", "country": "matched_country"}
    ),
    on="actor_key",
    how="left",
)

lookup_rows = pd.concat(
    [
        enterprise_country[["actor", "country"]].assign(source="enterprise"),
        competitor_country[["actor", "matched_country"]]
        .rename(columns={"matched_country": "country"})
        .assign(source="matched_competitor"),
    ],
    ignore_index=True,
)

def unique_or_na(values: pd.Series):
    cleaned = [str(v).strip() for v in values.dropna().unique() if str(v).strip()]
    if len(cleaned) == 1:
        return cleaned[0]
    return pd.NA

# Country attribution table for the plot: keep only unambiguous matches.
df_country_lookup = (
    lookup_rows.groupby("actor", as_index=False)
    .agg(country=("country", unique_or_na), matches=("source", "nunique"))
    .dropna(subset=["country"])
)

print(f"Country lookup table: {len(df_country_lookup)} actors matched")
df_country_lookup.head(10)

Country lookup table: 1672 actors matched


,actor,country,matches
0,01.AI,China,1
1,10AI GmbH,Germany,1
2,1X Technologies,United States,1
3,20face,Netherlands,1
4,2CRSI,France,1
5,36ZERO Vision,United Kingdom,1
6,3DUniversum,Netherlands,1
7,3dvisionlabs,Germany,1
8,3ive AI,Germany,1
9,4.screen,Germany,1


## 8. Visualisation 2D et export des artefacts

In [83]:
import plotly.express as px

info_cols = [
    "founded_year", "employees_count", "revenue_millions",
    "capitalization", "funds_raised", "description"
]
available = [c for c in info_cols if c in df_raw.columns]
df_info = df_raw.set_index(COL_NAME)[available]

df_plot = df_coords.copy()
for col in available:
    df_plot[col] = df_plot["actor"].map(df_info[col])

# Couleurs par pays: fallback SQL si la colonne country n'est pas dans df_raw.
if "country" in df_raw.columns:
    country_map = df_raw.set_index(COL_NAME)["country"]
else:
    with sqlite3.connect(DB_PATH) as con:
        df_country = pd.read_sql_query(
            f"SELECT name AS {COL_NAME}, country FROM enterprises",
            con,
        )
    country_map = df_country.set_index(COL_NAME)["country"]

df_plot["country"] = df_plot["actor"].map(country_map).fillna("Unknown")

def fmt_hover(row):
    lines = [f"<b>{row['actor']}</b>"]
    if pd.notna(row.get("sector")):
        lines.append(f"Sector: {row['sector']}")
    if pd.notna(row.get("country")):
        lines.append(f"Country: {row['country']}")
    if pd.notna(row.get("founded_year")):
        lines.append(f"Founded: {int(row['founded_year'])}")
    if pd.notna(row.get("employees_count")):
        lines.append(f"Employees: {int(row['employees_count']):,}")
    cap = float(row.get("capitalization") or 0)
    if cap > 0:
        lines.append(f"Market cap: {cap/1000:.1f}B USD")
    rev = float(row.get("revenue_millions") or 0)
    if rev > 0:
        lines.append(f"Revenue: {rev/1000:.1f}B USD")
    if pd.notna(row.get("description")):
        desc = str(row["description"])
        snippet = desc[:160].rstrip()
        lines.append(f"<i>{snippet}{'…' if len(desc) > 160 else ''}</i>")
    lines.append(f"Outgoing competitor links: {int(row['score'])}")
    return "<br>".join(lines)

df_plot["hover"] = df_plot.apply(fmt_hover, axis=1)
df_plot["marker_size"] = np.maximum(df_plot["score"] * 2, 4)

fig = px.scatter(
    df_plot,
    x="x", y="y",
    color="country",
    size="marker_size",
    size_max=24,
    text="actor",
    custom_data=["hover"],
    color_discrete_sequence=px.colors.qualitative.Light24,
    title=f"{method} — Entreprises dans l'espace des compétiteurs ({N} entreprises)",
    labels={"x": "Dimension 1", "y": "Dimension 2", "country": "Pays"},
    width=1400,
    height=1200,
)

fig.update_traces(
    hovertemplate="%{customdata[0]}<extra></extra>",
    textposition="top center",
    textfont=dict(size=7),
    marker=dict(opacity=0.8, line=dict(width=0.4, color="white")),
)

fig.update_layout(
    legend=dict(title="Pays", font=dict(size=10)),
    font=dict(family="Inter, sans-serif", size=11),
    plot_bgcolor="#FDFAF4",
    paper_bgcolor="#FDFAF4",
    hovermode="closest",
)

fig_html = EXPORTS_DIR / "competition_map_2d.html"
fig.write_html(str(fig_html))
print(f"Carte interactive ({method}) → {fig_html}")
fig.show()

Carte interactive (t-SNE) → C:\Users\33623\Documents\___Projets\AI\Reseaux d'acteurs\analyses\exports\competition_map_2d.html


In [84]:
print("── Récapitulatif des exports ────────────────────────────")
for p in [raw_path, long_path, agg_path, cooc_path, coords_path, fig_html]:
    size_kb = Path(p).stat().st_size / 1024
    print(f"  {p.name:<42} {size_kb:6.1f} KB")

── Récapitulatif des exports ────────────────────────────
  competitors_raw.csv                           9.3 KB
  competitors_long.csv                         26.1 KB
  competitors_aggregated.csv                   10.8 KB
  cooccurrence_matrix.csv                     108.1 KB
  coords_2d.csv                                 5.7 KB
  competition_map_2d.html                    4771.0 KB
